In [195]:
import os
import re
import glob
import math
import numpy as np
import pandas as pd
import neurokit2 as nk

In [196]:
# File Paths
BASE = "/Users/debbiehsu/Documents/PythonProjects/GTRI_MHMUAV"

# --- Part 1 inputs/outputs ---
XDF_FOLDER = f"{BASE}/Data"
METADATA_CSV = f"{BASE}/Metadata.csv"
RAW_PATH = f"{BASE}/physio_raw.csv"

# --- Part 2 output ---
CLEAN_PATH = f"{BASE}/physio_clean.csv"

# --- Part 3 outputs ---
EPOCH_PATH = f"{BASE}/epoch_availability.csv"
EDA_SCREEN_PATH = f"{BASE}/eda_record_screens.csv"
EDA_FLAT_PATH = f"{BASE}/eda_flat_runs.csv"

# --- Part 4 inputs ---
REC_META = f"{BASE}/Eye-tracking/Recording ID Metadata.csv"
FIXATIONS = f"{BASE}/Eye-tracking/UAV Mapper/fixations.csv"
AOI_FIXATIONS = f"{BASE}/Eye-tracking/UAV Mapper/aoi_fixations.csv"
AOI_LABELS = ["North", "South"]
FACE_FIXATIONS = f"{BASE}/Eye-tracking/Face Mapper/fixations_on_face.csv"
SURVEY_CSV = f"{BASE}/mdmt_physio.csv"
SAC_MAX_DUR_MS = None      # no duration filter for now; revisit later

CLOUD_ROOT = "/Users/debbiehsu/Library/CloudStorage/OneDrive-GeorgiaInstituteofTechnology/Documents/MHMUAV/R Code/Pupil Cloud"
PUPIL_DIR = f"{CLOUD_ROOT}/PupilClean"

# --- Part 4 output ---
FINAL_PATH = f"{BASE}/physio_eye_survey.csv"

In [197]:
PPG_FS = 25  # pulse sampling rate, Hz
EPOCH_SEC = 10  # epoch length for quality scoring
EPOCH_LEN = PPG_FS * EPOCH_SEC  # = 250 samples per epoch
PPG_CHANNELS = ("PPG_GRN_C", "PPG_GRN_IS")  # which pulse columns to clean

HR_MIN, HR_MAX = 30, 180  # plausible heart rate, bpm
IBI_MAX_SEC = 3.0  # longest plausible beat-to-beat gap
IBI_RATIO_MAX = 2.2  # longest gap / shortest gap
QUALITY_MIN = 0.86  # template-match quality floor

EDA_FS = 15  # electrodermal sampling rate, Hz
EDA_SENTINEL = 10000.0  # EmotiBit "no reading" code
EDA_CEILING_uS = 40.0  # physiologically implausible above this
EDA_TOL_SEC = 0.034  # merge tolerance, just over 1/2 of 1/15 s
EDA_MIN_RANGE_uS = 0.005  # dynamic-range floor for the screen
EDA_ROLE_FILE = {"EDA_C": "EDA_C.csv", "EDA_IS": "EDA_IS.csv"}

TASK_LABELS = ("HH", "LL", "M")
VIDEO_ORDERS = {
    1: ("LL", "HH", "M"), 2: ("LL", "M", "HH"), 3: ("HH", "LL", "M"),
    4: ("HH", "M", "LL"), 5: ("M", "LL", "HH"), 6: ("M", "HH", "LL"),
}
RESTART_FIXES = {25: 2, 46: 3}  # team -> which block was replayed
TOLERANCE = 0.021  # general merge tolerance, seconds

KEEP_ON_RESTART = {"ID", "Condition", "timestamp", "DataSyncMarker",
                   "DataSyncMarker_UNIX", "Reliability", "unix_ns", "dt_utc"}

CSV_DTYPES = {
    "ID": "int64", "unix_ns": "Int64", "DataSyncMarker": "float64",
    "accept": "boolean", "eda_ok": "boolean", "Fix_ID": "Int64", "Sac_ID": "Int64",
    "epoch": "int64", "n_peaks": "int64", "unix_start": "Int64",
}
CSV_DATES = ["dt_utc"]

In [198]:
# Settings
# --- PPG ---
PPG_FS = 25 # Sampling rate
EPOCH_SEC = 10 # Quality check window length: 10s (Vorreuther et al. 2025)
EPOCH_LEN = PPG_FS * EPOCH_SEC # = 250 samples per epoch
PPG_CHANNELS = ("PPG_GRN_C", "PPG_GRN_IS")

# PPG Quality Validation: Four criterion gate (Orphanidou et al., 2015)
HR_MIN, HR_MAX = 30, 180 # Plausible heart rate, beats per minute
IBI_MAX_SEC = 3.0 # No gap between beats longer than 3s
IBI_RATIO_MAX = 2.2 # Longest gap < 2.2x the shortest
QUALITY_MIN = 0.86 # Template-match correlation floor

# --- EDA ---
EDA_FS = 15 # Sampling rate
EDA_SENTINEL = 10000.0 # EmotiBit device failure code, should be removed from the data
EDA_CEILING_uS = 40.0 # Sanity ceiling
EDA_TOL_SEC = 0.034 # Tolerance second, half of the 15ms spacing
EDA_MIN_RANGE_uS = 0.005 # Selected for this data only - p5-p95 dynamic range screen
EDA_ROLE_FILE = {"EDA_C": "EDA_C.csv", "EDA_IS": "EDA_I.csv"}

# --- Study Design
TASK_LABELS = ("HH", "LL", "M") # Labels that represent task blocks
VIDEO_ORDERS = {
    1: ("LL", "HH", "M"), 2: ("LL", "M", "HH"), 3: ("HH", "LL", "M"),
    4: ("HH", "M", "LL"), 5: ("M", "LL", "HH"), 6: ("M", "HH", "LL"),
}
RESTART_FIXES = {25: 2, 46: 3} # Team -> which block number was restarted
TOLERANCE = 0.021 # Nearest-timestamp window for non-base streams

# Part 1 - XDF to 25Hz grid with UNIX Time

In [199]:
# Export each XDF stream to its own CSV
def export_streams_to_csv(streams, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    SUFFIX = {"MD-V5-0001023": "_C", "MD-V6-0000405": "_IS"} # EmotiBit serial number for each role
    written = []

    for stream in streams:
        name = stream["info"]["name"][0]
        if name == "Keyboard" or "Neon" in name:
            continue

        source_list = stream["info"].get("source_id") or ["Unknown"]
        source_id = source_list[0] or "Unknown"
        suffix = SUFFIX.get(source_id, "")

        values = np.asarray(stream["time_series"])
        times = stream["time_stamps"]
        if len(values) == 0:
            continue

        if values.ndim == 1 or values.shape[1] == 1: # Single channel stream
            table = pd.DataFrame({"timestamp": times, "value": np.ravel(values)})
        else: # Multi-channel, e.g. DataSyncMarker
            cols = [f"channel_{i}" for i in range(values.shape[1])]
            table = pd.DataFrame(values, columns=cols)
            table.insert(0, "timestamp", times) # Put timestamp first

        table.to_csv(os.path.join(output_dir, f"{name}{suffix}.csv"), index=False)
        written.append(name + suffix)
    return written

In [200]:
# Turn the DataSyncMarker channel into reliability labels
def label_reliability(marker, video_condition):
    """Marker states (1=video, 2=no video, 3=skip) -> HH / LL / M / No Video"""
    order = VIDEO_ORDERS.get(video_condition)
    if order is None:
        return pd.Series([None] * len(marker), index = marker.index)

    labels = []
    block_count = 0
    previous = None

    for value in marker:
        state = int(value) if  pd.notna(value) else None
        if state != previous:
            if state == 1:
                block_count += 1
        previous = state

        if state == 1 and 1 <= block_count <= len(order):
            labels.append(order[block_count - 1])
        elif state == 2:
            labels.append("No Video")
        else:
            labels.append(None)
    return pd.Series(labels, index=marker.index)

def attach_datasync(grid, datasync_path, video_condition):
    ds = pd.read_csv(datasync_path).sort_values("timestamp").reset_index(drop=True)
    ds["timestamp"] = ds["timestamp"].round(3) # Millisecond rounding

    if "channel_0" in ds.columns: # Marker state stream
        grid = pd.merge_asof(grid, ds[["timestamp", "channel_0"]],
                             on="timestamp", direction="backward")
        grid = grid.rename(columns={"channel_0": "DataSyncMarker"})
        grid["Reliability"] = label_reliability(grid["DataSyncMarker"], video_condition)

    if "channel_1" in ds.columns: # UNIX time stream
        grid = pd.merge_asof(grid, ds[["timestamp", "channel_1"]],
                             on="timestamp", direction="backward")
        grid = grid.rename(columns={"channel_1": "DataSyncMarker_UNIX"})
    return grid

In [201]:
# Column Names
def tidy_name(name):
    new = name[:-6] if name.endswith("_value") else name
    if new.endswith("_I"):
        new = new + "S"
    new = new.replace("TEMP1_", "TEMP_")
    return new


def tidy_column_names(grid):
    return grid.rename(columns={c: tidy_name(c) for c in grid.columns})


def order_columns(grid):
    desired = ["ID", "Condition", "timestamp",
               "DataSyncMarker", "DataSyncMarker_UNIX", "Reliability",
               "PPG_GRN_C", "PPG_RED_C", "PPG_IR_C", "EDA_C", "HR_C", "TEMP_C", "THERM_C",
               "PPG_GRN_IS", "PPG_RED_IS", "PPG_IR_IS", "EDA_IS", "HR_IS", "TEMP_IS",
               "THERM_IS"]
    front = [c for c in desired if c in grid.columns]
    rest = [c for c in grid.columns if c not in front]
    return grid[front + rest]

In [202]:
def merge_raw(input_dir, team_id, condition, video_condition,
              tolerance=TOLERANCE, grid_fs=PPG_FS, verbose=True):
    all_files = glob.glob(os.path.join(input_dir, "*.csv"))
    base_file = None
    datasync_file = None
    other_files = []

    for path in all_files:
        name = os.path.basename(path)
        if "PPG_GRN" in name and name.endswith("_C.csv"):
            base_file = path
        elif "DataSyncMarker" in name:
            datasync_file = path
        elif "Neon" in name or "Keyboard" in name:
            continue
        else:
            other_files.append(path)

    if base_file is None:
        raise FileNotFoundError(f"No PPG_GRN_*_C.csv in {input_dir}")

    base = pd.read_csv(base_file).sort_values("timestamp").reset_index(drop=True)
    base["timestamp"] = base["timestamp"].round(3)

    grid = pd.DataFrame({"ID": team_id, "Condition": condition,
                         "timestamp": base["timestamp"]})
    if datasync_file is not None:
        grid = attach_datasync(grid, datasync_file, video_condition)
    grid["PPG_GRN_C"] = base["value"].to_numpy()

    fill_limits = {}
    dt_grid = 1.0 / grid_fs
    for path in sorted(other_files):
        stem = os.path.basename(path)[:-4]
        stream = pd.read_csv(path)
        if "timestamp" not in stream.columns:
            continue
        stream["timestamp"] = stream["timestamp"].round(3)
        stream = stream.sort_values("timestamp").reset_index(drop=True)
        value_col = "value" if "value" in stream.columns else stream.columns[1]
        stream = stream[["timestamp", value_col]].rename(columns={value_col: stem})

        dt_native = stream["timestamp"].diff().median()
        if pd.isna(dt_native) or dt_native <= 0:
            limit = 1
        else:
            limit = max(1, int(math.ceil(dt_native / dt_grid)) - 1)
        fill_limits[tidy_name(stem)] = limit

        grid = pd.merge_asof(grid, stream, on="timestamp",
                             direction="nearest", tolerance=tolerance)

    grid = grid.sort_values("timestamp").reset_index(drop=True)
    grid = tidy_column_names(grid)
    for column, limit in fill_limits.items():
        if column in grid.columns:
            grid[column] = grid[column].ffill(limit=limit)
    if verbose and fill_limits:
        shown = ", ".join(f"{c}:{l}" for c, l in sorted(fill_limits.items()))
        print(f"  fill limits (rows) -> {shown}")
    return order_columns(grid)

In [203]:
# Restart Handling
def reblank_restart(grid):
    if "Reliability" not in grid.columns:
        return grid
    blank_cols = [c for c in grid.columns if c not in KEEP_ON_RESTART and c != "seg_id"]
    mask = grid["Reliability"] == "Restart"
    if mask.any() and blank_cols:
        grid.loc[mask, blank_cols] = np.nan
    return grid


def label_restart(grid, restart_block, video_condition):
    grid = grid.reset_index(drop=True)
    order = VIDEO_ORDERS.get(video_condition)
    if order is None:
        return grid

    def to_state(v):
        return None if pd.isna(v) else int(round(v))

    states = grid["DataSyncMarker"].map(to_state)

    runs = []
    run_state = states.iloc[0]
    run_start = 0
    for i in range(1, len(states)):
        if states.iloc[i] != run_state:
            runs.append((run_state, run_start, i - 1))
            run_state, run_start = states.iloc[i], i
    runs.append((run_state, run_start, len(states) - 1))

    one_runs = [r for r in runs if r[0] == 1]
    three_runs = [r for r in runs if r[0] == 3]
    if len(one_runs) < restart_block or not three_runs:
        return grid

    delete_start = one_runs[restart_block - 1][1]
    delete_end = three_runs[-1][2]
    if delete_end <= delete_start:
        return grid

    labels = [None] * len(grid)
    block_count = 0
    previous_kept = None
    for i, state in enumerate(states):
        if delete_start <= i <= delete_end:
            labels[i] = "Restart"
            continue
        if state == 1:
            if previous_kept != 1:
                block_count += 1
            labels[i] = order[block_count - 1] if 1 <= block_count <= len(order) else None
        elif state == 2:
            labels[i] = "No Video"
        previous_kept = state

    grid["Reliability"] = labels
    return reblank_restart(grid)

In [204]:
# UNIX Timestamp
def add_unix_time(grid, marker_col="DataSyncMarker_UNIX", clock_col="timestamp",
                  unix_lo=1e9, unix_hi=2e9, verbose=True):
    pieces = []
    for team_id, one in grid.groupby("ID", sort=True):
        one = one.sort_values(clock_col).reset_index(drop=True)
        marker = pd.to_numeric(one[marker_col], errors="coerce")
        valid = marker.notna() & marker.between(unix_lo, unix_hi)
        is_new = valid & (marker != marker.shift())

        if not is_new.any():
            one["unix_ns"] = pd.array([pd.NA] * len(one), dtype="Int64")
            pieces.append(one)
            if verbose:
                print(f"  ID {team_id}: no valid sync markers")
            continue

        anchor_clock = one.loc[is_new, clock_col].to_numpy(float)
        anchor_offset = marker[is_new].to_numpy(float) - anchor_clock
        order = np.argsort(anchor_clock)
        anchor_clock, anchor_offset = anchor_clock[order], anchor_offset[order]

        clock = one[clock_col].to_numpy(float)
        offset = np.interp(clock, anchor_clock, anchor_offset)
        one["unix_ns"] = pd.array(np.round((clock + offset) * 1e9).astype("int64"),
                                  dtype="Int64")
        pieces.append(one)
        if verbose:
            spread_ms = (anchor_offset.max() - anchor_offset.min()) * 1000
            print(f"  ID {team_id}: {is_new.sum()} anchors, offset spread {spread_ms:.1f} ms")

    out = pd.concat(pieces, ignore_index=True)
    out["dt_utc"] = pd.to_datetime(out["unix_ns"].astype("float"), unit="ns", utc=True)
    return out

In [205]:
# Confirm Restart Fix
def check_task_runs(grid, restart_fixes=RESTART_FIXES, expected=3):
    rows = []
    for team_id, one in grid.groupby("ID", sort=True):
        state = one["DataSyncMarker"]
        n_runs = int(((state == 1) & (state.shift() != 1)).sum())
        declared = int(team_id) in restart_fixes
        rows.append({"ID": int(team_id), "task_runs": n_runs,
                     "declared_restart": declared,
                     "labels_trustworthy": (n_runs == expected) or declared})
    return pd.DataFrame(rows)

In [206]:
# Loading Metadata
def load_meta(path, need=("video_condition", "condition")):
    m = pd.read_csv(path)
    m = m.loc[:, ~m.columns.duplicated()]
    m = m.loc[:, ~m.columns.str.startswith("Unnamed:")]
    m = m.dropna(axis=1, how="all")
    for c in m.columns:
        if m[c].dtype == object:
            m[c] = m[c].str.strip()
    have = [c for c in need if c in m.columns]
    counts = m.groupby("Team_ID")[have].nunique()
    bad = counts[(counts > 1).any(axis=1)]
    assert bad.empty, f"Metadata.csv disagrees within team:\n{bad}"
    return m.drop_duplicates(subset="Team_ID").set_index("Team_ID")

In [207]:
# Part 1 Execution Cell
meta = load_meta(METADATA_CSV)
assert meta.index.is_unique

EXCLUDE = {3, 5, 8, 11, 13, 59}

teams = [t for t in sorted(meta.index) if t not in EXCLUDE]
found = [t for t in teams
         if os.path.exists(os.path.join(XDF_FOLDER, f"{t}_Data"))
         or os.path.exists(os.path.join(XDF_FOLDER, f"{t}.xdf"))]
print(f"{len(meta)} in metadata, {len(EXCLUDE)} excluded, "
      f"{len(teams) - len(found)} with no source data: "
      f"{sorted(set(teams) - set(found))}")
teams = found

pieces = []
for tid in teams:
    print(f"ID {tid}")
    folder = os.path.join(XDF_FOLDER, f"{tid}_Data")
    vid = int(meta.loc[tid, "video_condition"])

    # export from .xdf only if this team hasn't been exported yet
    if not os.path.exists(folder):
        import pyxdf

        streams, _ = pyxdf.load_xdf(os.path.join(XDF_FOLDER, f"{tid}.xdf"))
        export_streams_to_csv(streams, folder)

    g = merge_raw(folder, tid, meta.loc[tid, "condition"], vid)
    if tid in RESTART_FIXES:
        g = label_restart(g, RESTART_FIXES[tid], vid)
    pieces.append(g)

raw = add_unix_time(pd.concat(pieces, ignore_index=True))
save_table(raw, RAW_PATH)

runs = check_task_runs(raw)
print(runs.to_string(index=False))
assert runs["labels_trustworthy"].all(), (
    "task-run count wrong for teams: "
    f"{runs.loc[~runs['labels_trustworthy'], 'ID'].tolist()} - "
    "add each to RESTART_FIXES with the block number that was replayed")
print(raw["Reliability"].value_counts(dropna=False).to_dict())

57 in metadata, 6 excluded, 0 with no source data: []
ID 4
  fill limits (rows) -> EDA_C:1, EDA_IS:1, HR_C:19, HR_IS:27, PPG_GRN_IS:1, PPG_IR_C:1, PPG_IR_IS:1, PPG_RED_C:1, PPG_RED_IS:1, TEMP_C:3, TEMP_IS:3, THERM_C:3, THERM_IS:3
ID 6
  fill limits (rows) -> EDA_C:1, EDA_IS:1, HR_C:22, HR_IS:24, PPG_GRN_IS:1, PPG_IR_C:1, PPG_IR_IS:1, PPG_RED_C:1, PPG_RED_IS:1, TEMP_C:3, TEMP_IS:3, THERM_C:3, THERM_IS:3
ID 7
  fill limits (rows) -> EDA_C:1, EDA_IS:1, HR_C:20, HR_IS:27, PPG_GRN_IS:1, PPG_IR_C:1, PPG_IR_IS:1, PPG_RED_C:1, PPG_RED_IS:1, TEMP_C:3, TEMP_IS:3, THERM_C:3, THERM_IS:3
ID 10
  fill limits (rows) -> EDA_C:1, EDA_IS:1, HR_C:29, HR_IS:30, PPG_GRN_IS:1, PPG_IR_C:1, PPG_IR_IS:1, PPG_RED_C:1, PPG_RED_IS:1, TEMP_C:3, TEMP_IS:3, THERM_C:3, THERM_IS:3
ID 12
  fill limits (rows) -> EDA_C:1, EDA_IS:1, HR_C:20, HR_IS:21, PPG_GRN_IS:1, PPG_IR_C:1, PPG_IR_IS:1, PPG_RED_C:1, PPG_RED_IS:1, TEMP_C:3, TEMP_IS:3, THERM_C:3, THERM_IS:3
ID 14
  fill limits (rows) -> EDA_C:1, EDA_IS:1, HR_C:21, HR_IS:

# Part 2 - Cleaning

In [208]:
# Creating clean segments (contiguous stretch of one label for one team)
def add_segment_id(grid):
    grid = grid.sort_values(["ID", "timestamp"]).reset_index(drop=True)
    label = grid["Reliability"].fillna("__none__")
    changed = (label != label.shift()) | (grid["ID"] != grid["ID"].shift())
    grid["seg_id"] = changed.cumsum()
    return grid


def finite_runs(values, min_length):
    good = np.isfinite(values)
    padded = np.concatenate(([0], good.astype(int), [0]))
    change = np.diff(padded)
    starts = np.where(change == 1)[0]
    ends = np.where(change == -1)[0]
    return [(s, e) for s, e in zip(starts, ends) if (e - s) >= min_length]

In [209]:
# PPG Cleaning
def clean_ppg_segment(values, fs=PPG_FS, min_length=EPOCH_LEN):
    out = np.full(len(values), np.nan)
    for start, end in finite_runs(values, min_length):
        out[start:end] = nk.ppg_clean(values[start:end], sampling_rate=fs,
                                      method="langevin2021")
    return out


def clean_ppg(grid, channels=PPG_CHANNELS, fs=PPG_FS, verbose=True):
    grid = add_segment_id(grid)
    for channel in channels:
        if channel not in grid.columns:
            continue
        cleaned = np.full(len(grid), np.nan)
        for _, segment in grid.groupby("seg_id", sort=False):
            rows = segment.index.to_numpy()
            cleaned[rows] = clean_ppg_segment(segment[channel].to_numpy(float), fs=fs)
        grid[channel + "_clean"] = cleaned
        if verbose:
            n = np.isfinite(cleaned).sum()
            print(f"  {channel}: filtered {n:,}/{len(grid):,} samples "
                  f"({100 * n / len(grid):.1f}%)")
    return grid

In [210]:
# EDA Cleaning
def clean_eda_source(path, fs=EDA_FS, sentinel=EDA_SENTINEL):
    raw = pd.read_csv(path).sort_values("timestamp").reset_index(drop=True)
    raw["timestamp"] = raw["timestamp"].round(3)
    values = raw["value"].to_numpy(float)

    n_sentinel = int((values >= sentinel).sum())
    values[values >= sentinel] = np.nan

    values = pd.Series(values).interpolate(limit_direction="both").to_numpy()
    if not np.isfinite(values).all():
        return None, n_sentinel

    cleaned = nk.eda_clean(values, sampling_rate=fs)
    return pd.DataFrame({"timestamp": raw["timestamp"], "eda": cleaned}), n_sentinel


def clean_eda(grid, xdf_folder, role_file=EDA_ROLE_FILE, tol=EDA_TOL_SEC,
              ceiling=EDA_CEILING_uS, fill_limit=1, verbose=True):
    grid = grid.sort_values(["ID", "timestamp"]).reset_index(drop=True)

    for column, filename in role_file.items():
        pieces = []
        total_sentinel = 0
        missing = []
        for team_id in sorted(grid["ID"].unique()):
            path = os.path.join(xdf_folder, f"{team_id}_Data", filename)
            if not os.path.exists(path):
                missing.append(team_id)
                continue
            block, n_sentinel = clean_eda_source(path)
            total_sentinel += n_sentinel
            if block is None:
                continue
            block.insert(0, "ID", team_id)
            pieces.append(block.rename(columns={"eda": column + "_clean"}))

        if missing:
            print(f"  WARNING: no {filename} for {len(missing)} team(s): "
                  f"{[int(t) for t in missing[:10]]}")
        if not pieces:
            print(f"  WARNING: {column}_clean NOT CREATED - no source files found")
            continue

        eda = pd.concat(pieces, ignore_index=True).sort_values("timestamp")
        eda = eda.reset_index(drop=True)
        grid = grid.sort_values("timestamp").reset_index(drop=True)
        grid = pd.merge_asof(grid, eda, on="timestamp", by="ID",
                             direction="nearest", tolerance=tol)

        new_col = column + "_clean"
        matched = int(grid[new_col].notna().sum())
        grid[new_col] = grid.groupby("ID")[new_col].ffill(limit=fill_limit)
        n_over = int((grid[new_col] > ceiling).sum())
        grid.loc[grid[new_col] > ceiling, new_col] = np.nan
        if verbose:
            filled = int(grid[new_col].notna().sum())
            print(f"  {new_col}: {matched:,} matched directly, "
                  f"{filled - matched:,} held (limit {fill_limit}), "
                  f"{total_sentinel} sentinel removed, {n_over} above {ceiling} uS")
    return grid.sort_values(["ID", "timestamp"]).reset_index(drop=True)

In [211]:
# Part 2 Execution
clean = clean_ppg(raw)
clean = clean_eda(clean, XDF_FOLDER)
clean = reblank_restart(clean)
save_table(clean.drop(columns=["seg_id"]), CLEAN_PATH)

print("restart rows with sensor data:",
      int(clean.loc[clean["Reliability"] == "Restart",
      [c for c in clean.columns
       if c not in KEEP_ON_RESTART and c != "seg_id"]].notna().sum().sum()))

  PPG_GRN_C: filtered 4,011,392/4,023,825 samples (99.7%)
  PPG_GRN_IS: filtered 3,970,886/4,023,825 samples (98.7%)
  EDA_C_clean: 4,023,766 matched directly, 15 held (limit 1), 61 sentinel removed, 5 above 40.0 uS
  EDA_IS_clean: 3,983,291 matched directly, 19 held (limit 1), 80 sentinel removed, 0 above 40.0 uS
 wrote 4,023,825 rows -> /Users/debbiehsu/Documents/PythonProjects/GTRI_MHMUAV/physio_clean.csv
restart rows with sensor data: 0


# Part 3 - Epoch Quality and Screen

In [212]:
# Scoring one epoch
def score_epoch(values, fs=PPG_FS):
    result = {"n_peaks": 0, "hr_bpm": np.nan, "quality": np.nan, "accept": False}
    if not np.isfinite(values).all():
        return result
    try:
        peaks = nk.ppg_findpeaks(values, sampling_rate=fs, method="bishop")["PPG_Peaks"]
    except Exception:
        return result
    result["n_peaks"] = len(peaks)
    if len(peaks) < 3:
        return result
    ibi = np.diff(peaks) / fs
    hr = 60.0 / np.median(ibi)
    result["hr_bpm"] = hr
    try:
        quality = float(np.nanmean(
            nk.ppg_quality(values, peaks=peaks, sampling_rate=fs, method="templatematch")))
    except Exception:
        return result
    result["quality"] = quality
    result["accept"] = bool(
        (HR_MIN <= hr <= HR_MAX)
        and (ibi.max() <= IBI_MAX_SEC)
        and (ibi.max() / ibi.min() < IBI_RATIO_MAX)
        and (quality >= QUALITY_MIN))
    return result

In [213]:
# Producing Epoch Table
def build_epoch_table(grid, roles=("C", "IS"), fs=PPG_FS, epoch_len=EPOCH_LEN):
    rows = []
    task = grid[grid["Reliability"].isin(TASK_LABELS)]
    for (team_id, reliability), block in task.groupby(["ID", "Reliability"], sort=True):
        block = block.sort_values("timestamp")
        for role in roles:
            ppg_col = f"PPG_GRN_{role}_clean"
            eda_col = f"EDA_{role}_clean"
            if ppg_col not in block.columns:
                continue
            ppg = block[ppg_col].to_numpy(float)
            eda = (block[eda_col].to_numpy(float) if eda_col in block.columns
                   else np.full(len(block), np.nan))
            unix = block["unix_ns"].to_numpy()
            for k in range(len(ppg) // epoch_len):
                lo, hi = k * epoch_len, (k + 1) * epoch_len
                scored = score_epoch(ppg[lo:hi], fs=fs)
                chunk = eda[lo:hi]
                has_eda = bool(np.isfinite(chunk).any())
                rows.append({
                    "ID": team_id, "Reliability": reliability, "Role": role, "epoch": k,
                    "unix_start": unix[lo] if pd.notna(unix[lo]) else pd.NA,
                    "n_peaks": scored["n_peaks"], "hr_bpm": scored["hr_bpm"],
                    "quality": scored["quality"], "accept": scored["accept"],
                    "eda_mean": np.nanmean(chunk) if has_eda else np.nan,
                    "eda_sd": np.nanstd(chunk) if has_eda else np.nan,
                    "eda_median": np.nanmedian(chunk) if has_eda else np.nan,
                    "eda_coverage": float(np.isfinite(chunk).mean()),
                })
    return pd.DataFrame(rows)

In [214]:
# EDA Screening (on raw EDA)
def eda_range_screen(grid, roles=("C", "IS"), min_range=EDA_MIN_RANGE_uS):
    rows = []
    task = grid[grid["Reliability"].isin(TASK_LABELS)]
    for (team_id, reliability), block in task.groupby(["ID", "Reliability"], sort=True):
        for role in roles:
            col = f"EDA_{role}_clean"
            if col not in block.columns:
                continue
            values = block[col].to_numpy(float)
            values = values[np.isfinite(values)]
            if len(values) == 0:
                rows.append({"ID": team_id, "Reliability": reliability, "Role": role,
                             "p5_p95_range": np.nan, "eda_ok": False})
                continue
            spread = np.percentile(values, 95) - np.percentile(values, 5)
            rows.append({"ID": team_id, "Reliability": reliability, "Role": role,
                         "p5_p95_range": spread, "eda_ok": bool(spread >= min_range)})
    return pd.DataFrame(rows)


def longest_flat_run(values, fs=PPG_FS):
    v = pd.Series(values).to_numpy(float)
    if len(v) == 0:
        return 0.0
    same = np.concatenate(([False], v[1:] == v[:-1]))
    run = best = 0
    for s in same:
        run = run + 1 if s else 0
        best = max(best, run)
    return (best + 1) / fs


def eda_flat_report(grid, roles=("C", "IS"), fs=PPG_FS):
    rows = []
    task = grid[grid["Reliability"].isin(TASK_LABELS)]
    for (team_id, reliability), block in task.groupby(["ID", "Reliability"], sort=True):
        for role in roles:
            col = f"EDA_{role}"
            if col not in block.columns:
                continue
            v = block[col].to_numpy(float)
            v = v[np.isfinite(v)]
            if len(v) == 0:
                rows.append({"ID": team_id, "Reliability": reliability, "Role": role,
                             "block_sec": len(block) / fs, "flat_run_sec": np.nan,
                             "median_uS": np.nan, "p5_p95_range": np.nan})
                continue
            rows.append({
                "ID": team_id, "Reliability": reliability, "Role": role,
                "block_sec": len(block) / fs,
                "flat_run_sec": longest_flat_run(v, fs=fs),
                "median_uS": float(np.median(v)),
                "p5_p95_range": float(np.percentile(v, 95) - np.percentile(v, 5)),
            })
    out = pd.DataFrame(rows)
    if len(out):
        out["flat_frac"] = out["flat_run_sec"] / out["block_sec"]
        out = out.sort_values("flat_frac", ascending=False).reset_index(drop=True)
    return out

In [215]:
# Part 3 Execution
epochs = build_epoch_table(clean)
screen = eda_range_screen(clean)
flat = eda_flat_report(raw)  # raw, not clean - see Cell 20

save_table(epochs, EPOCH_PATH)
save_table(screen, EDA_SCREEN_PATH)
save_table(flat, EDA_FLAT_PATH)

print("accept rate:", epochs.groupby("Reliability")["accept"].mean().round(3).to_dict())
print("eda_coverage by role:", epochs.groupby("Role")["eda_coverage"].mean().round(3).to_dict())
print("screen failures:", int((~screen["eda_ok"]).sum()), "of", len(screen))
print(flat.head(6).to_string(index=False))

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/neurokit2/signal/signal_period.py:83: NeuroKitWarning: Too few peaks detected to compute the rate. Returning empty vector.
  warn(


 wrote 18,914 rows -> /Users/debbiehsu/Documents/PythonProjects/GTRI_MHMUAV/epoch_availability.csv
 wrote 306 rows -> /Users/debbiehsu/Documents/PythonProjects/GTRI_MHMUAV/eda_record_screens.csv
 wrote 306 rows -> /Users/debbiehsu/Documents/PythonProjects/GTRI_MHMUAV/eda_flat_runs.csv
accept rate: {'HH': 0.545, 'LL': 0.511, 'M': 0.55}
eda_coverage by role: {'C': 1.0, 'IS': 0.992}
screen failures: 73 of 306
 ID Reliability Role  block_sec  flat_run_sec  median_uS  p5_p95_range  flat_frac
 31           M    C     607.92         62.08   0.030147      0.000003   0.102119
 28          HH    C     621.68         51.24   0.030145      0.010592   0.082422
 54          LL   IS     610.20         44.72   0.030149      0.029017   0.073287
 34           M    C     605.96         44.16   0.030154      0.000040   0.072876
 19          LL    C     607.48         42.44   0.030148      0.000003   0.069862
 34          LL   IS     609.40         42.40   0.030146      0.000156   0.069577


# Part 4 - Eyetracking & Survey Merging

In [216]:
# Fixation Table
def norm_cols(df):
    df = df.copy()
    df.columns = df.columns.str.strip().str.lower()
    return df


def build_fixation_table(fixations_path, aoi_path, metadata_path, aoi_labels,
                         face_path=None):
    fix = norm_cols(pd.read_csv(fixations_path))
    aoi = norm_cols(pd.read_csv(aoi_path))
    meta = norm_cols(pd.read_csv(metadata_path))

    hits = pd.crosstab([aoi["recording id"], aoi["fixation id"]], aoi["aoi label"])
    hits = (hits > 0).astype("int8").reindex(columns=aoi_labels, fill_value=0)
    hits.columns = [f"AOI_{c}" for c in hits.columns]
    aoi_cols = list(hits.columns)
    hits = hits.reset_index()

    fix = fix.merge(hits, on=["recording id", "fixation id"], how="left")
    fix[aoi_cols] = fix[aoi_cols].fillna(0).astype("int8")
    fix = fix.merge(meta, on="recording id", how="left")

    fix = fix.rename(columns={
        "team id": "Team ID", "recording id": "Recording ID", "fixation id": "Fixation ID",
        "start timestamp [ns]": "Start", "end timestamp [ns]": "End",
        "duration [ms]": "Duration (ms)"})

    if face_path is not None:
        face = norm_cols(pd.read_csv(face_path))
        face = face[["recording id", "fixation id", "fixation on face"]].rename(columns={
            "recording id": "Recording ID", "fixation id": "Fixation ID",
            "fixation on face": "AOI_Face"})
        face["AOI_Face"] = face["AOI_Face"].astype(bool).astype("int8")
        fix = fix.merge(face, on=["Recording ID", "Fixation ID"], how="left")
        fix["AOI_Face"] = fix["AOI_Face"].fillna(0).astype("int8")
        aoi_cols = aoi_cols + ["AOI_Face"]

    keep = (["Team ID", "Recording ID", "Fixation ID", "Start", "End", "Duration (ms)"]
            + aoi_cols)
    fix = fix[keep].copy()
    fix = fix[fix["Team ID"].notna()]
    fix["Team ID"] = fix["Team ID"].astype("int64")
    fix["Start"] = fix["Start"].astype("int64")
    fix["End"] = fix["End"].astype("int64")
    return fix, aoi_cols

In [224]:
# Saccade Table
def build_saccade_table(cloud_root, metadata_path, max_dur_ms=None, verbose=True):
    meta = norm_cols(pd.read_csv(metadata_path))[["recording id", "team id"]]

    paths = sorted(glob.glob(os.path.join(cloud_root, "**", "saccades.csv"), recursive=True))
    if verbose:
        print(f"  found {len(paths)} saccades.csv file(s)")

    frames = [norm_cols(pd.read_csv(p)) for p in paths]
    frames = [f for f in frames if len(f)]
    if not frames:
        raise FileNotFoundError(f"No non-empty saccades.csv under {cloud_root!r} "
                                f"(isdir={os.path.isdir(cloud_root)})")

    sac = pd.concat(frames, ignore_index=True).merge(meta, on="recording id", how="left")
    sac = sac.rename(columns={
        "team id": "Team ID", "recording id": "Recording ID", "saccade id": "Saccade ID",
        "start timestamp [ns]": "Start", "end timestamp [ns]": "End",
        "duration [ms]": "Duration (ms)", "amplitude [px]": "Amplitude (px)",
        "amplitude [deg]": "Amplitude (deg)", "mean velocity [px/s]": "Mean velocity (px/s)",
        "peak velocity [px/s]": "Peak velocity (px/s)"})
    sac["Start"] = sac["Start"].astype("int64")
    sac["End"] = sac["End"].astype("int64")

    bad = sac[sac["Team ID"].isna()]
    if len(bad) and verbose:
        print(f"  DROPPED {len(bad):,} saccades from {bad['Recording ID'].nunique()} "
              f"recording(s) not in metadata: {sorted(bad['Recording ID'].unique())}")
    sac = sac[sac["Team ID"].notna()].copy()
    sac["Team ID"] = sac["Team ID"].astype("int64")

    n0 = len(sac)
    if max_dur_ms is not None:
        sac = sac[sac["Duration (ms)"] <= max_dur_ms].copy()
    if verbose:
        print(f"  {n0:,} saccades from {sac['Recording ID'].nunique()} recording(s)"
              + (f", {n0 - len(sac):,} dropped over {max_dur_ms} ms" if max_dur_ms is not None
                 else ", no duration filter applied"))
    return sac

In [218]:
# Attach events to grid
def add_join_key(grid):
    grid = grid.copy()
    grid["_unix_key"] = grid["unix_ns"].fillna(-1).astype("int64")
    return grid


def attach_fixations(grid, fix, aoi_cols):
    events = fix[["Team ID", "Start", "End", "Fixation ID", "Duration (ms)"] + aoi_cols]
    grid = add_join_key(grid).sort_values("_unix_key").reset_index(drop=True)
    events = events.sort_values("Start").reset_index(drop=True)
    joined = pd.merge_asof(grid, events, left_on="_unix_key", right_on="Start",
                           left_by="ID", right_by="Team ID", direction="backward")
    inside = (joined["_unix_key"] <= joined["End"]) & joined["End"].notna()
    joined["in_fixation"] = inside.astype("int8")
    for col in aoi_cols:
        joined[col] = joined[col].fillna(0).astype("int8") * inside.astype("int8")
    renames = {"Fixation ID": "Fix_ID", "Duration (ms)": "Fix_Duration_ms",
               "Start": "Fix_Start", "End": "Fix_End"}
    joined = joined.rename(columns=renames)
    joined.loc[~inside, list(renames.values())] = np.nan
    return joined.drop(columns=["Team ID", "_unix_key"])


def attach_saccades(grid, sac):
    renames = {"Saccade ID": "Sac_ID", "Duration (ms)": "Sac_Duration_ms",
               "Amplitude (px)": "Sac_Amplitude_px", "Amplitude (deg)": "Sac_Amplitude_deg",
               "Mean velocity (px/s)": "Sac_MeanVel_px_s",
               "Peak velocity (px/s)": "Sac_PeakVel_px_s",
               "Start": "Sac_Start", "End": "Sac_End"}
    cols = ["Team ID", "Start", "End"] + [c for c in renames if c not in ("Start", "End")]
    events = sac[[c for c in cols if c in sac.columns]]
    events = events.sort_values("Start").reset_index(drop=True)
    grid = add_join_key(grid).sort_values("_unix_key").reset_index(drop=True)
    joined = pd.merge_asof(grid, events, left_on="_unix_key", right_on="Start",
                           left_by="ID", right_by="Team ID", direction="backward")
    inside = (joined["_unix_key"] <= joined["End"]) & joined["End"].notna()
    joined["in_saccade"] = inside.astype("int8")
    joined = joined.rename(columns=renames)
    present = [v for v in renames.values() if v in joined.columns]
    joined.loc[~inside, present] = np.nan
    return joined.drop(columns=["Team ID", "_unix_key"])

In [219]:
# Pupil Binning
def bin_pupil(grid, pupil_dir, pattern="*_pupil_clean.parquet", tol_ns=int(20e6)):
    grid = grid.sort_values(["ID", "unix_ns"]).reset_index(drop=True)
    grid["_row"] = np.arange(len(grid), dtype="int64")
    frames = []
    for path in sorted(glob.glob(os.path.join(pupil_dir, pattern))):
        cols = ["subject", "ts_unix_s", "pupil_clean", "blink"]
        if path.endswith(".parquet"):
            frames.append(pd.read_parquet(path, columns=cols))
        else:
            frames.append(pd.read_csv(path, usecols=cols))
    if not frames:
        raise FileNotFoundError(f"No {pattern} in {pupil_dir}")
    pupil = pd.concat(frames, ignore_index=True).rename(columns={"subject": "ID"})
    pupil["ID"] = pupil["ID"].astype("int64")
    pupil["unix_ns"] = (pupil["ts_unix_s"] * 1e9).round().astype("int64")
    pupil["blink"] = pupil["blink"].astype(float)
    pupil = pupil.sort_values("unix_ns").reset_index(drop=True)

    target = grid.loc[grid["unix_ns"].notna(), ["ID", "unix_ns", "_row"]].copy()
    target["unix_ns"] = target["unix_ns"].astype("int64")
    target = target.sort_values("unix_ns").reset_index(drop=True)

    matched = pd.merge_asof(pupil, target, on="unix_ns", by="ID",
                            direction="nearest", tolerance=tol_ns)
    matched = matched.dropna(subset=["_row"])
    matched["_row"] = matched["_row"].astype("int64")
    summary = matched.groupby("_row").agg(
        Pupil_clean=("pupil_clean", "mean"),
        Pupil_blink_frac=("blink", "mean"),
        Pupil_n=("pupil_clean", "size"))
    grid = grid.merge(summary, left_on="_row", right_index=True, how="left")
    return grid.drop(columns=["_row"])

In [220]:
# Block numbers & Survey join
def add_block_number(grid):
    """Block 1/2/3 per team. Restart rows inherit the block that preceded them."""
    grid = grid.sort_values(["ID", "timestamp"]).reset_index(drop=True)
    grid["Block"] = 1

    for team_id, one in grid.groupby("ID"):
        keep = one["Reliability"] != "Restart"
        marker = one.loc[keep, "DataSyncMarker"]
        is_start = (marker == 1) & (marker.shift() != 1)
        block = is_start.cumsum().clip(lower=1)
        block = block.reindex(one.index).ffill().bfill().astype(int)
        grid.loc[one.index, "Block"] = block.to_numpy()
    return grid


def merge_survey(grid, survey):
    survey = survey.copy()
    for c in ("Role", "Condition"):
        if c in survey.columns:
            survey[c] = survey[c].str.strip()

    key_cols = {"Team_ID", "Role", "Condition", "Block", "Participant_ID"}
    role_suffix = {"Commander": "_C", "Intelligence Support": "_IS"}
    value_cols = [c for c in survey.columns if c not in key_cols]

    grid = add_block_number(grid)
    for role, suffix in role_suffix.items():
        part = survey[survey["Role"] == role].copy()
        part = part.rename(columns={"Team_ID": "ID",
                                    **{c: c + suffix for c in value_cols}})
        wanted = ["ID", "Condition", "Block"] + [c + suffix for c in value_cols]
        part = part[[c for c in wanted if c in part.columns]]
        grid = grid.merge(part, on=["ID", "Condition", "Block"], how="left")
    return grid.drop(columns=["Block"])

In [225]:
# Part 4 Execution
fix, aoi_cols = build_fixation_table(FIXATIONS, AOI_FIXATIONS, REC_META,
                                     AOI_LABELS, face_path=FACE_FIXATIONS)
sac = build_saccade_table(CLOUD_ROOT, REC_META, max_dur_ms=SAC_MAX_DUR_MS)

merged = attach_fixations(clean, fix, aoi_cols)
merged = attach_saccades(merged, sac)
merged = bin_pupil(merged, PUPIL_DIR)
merged = merge_survey(merged, pd.read_csv(SURVEY_CSV))
merged = reblank_restart(merged)

if {"Video_Condition_C", "Video_Condition_IS"} <= set(merged.columns):
    merged["Video_Condition"] = merged["Video_Condition_C"].fillna(merged["Video_Condition_IS"])
    merged = merged.drop(columns=["Video_Condition_C", "Video_Condition_IS"])

save_table(merged.drop(columns=["seg_id"], errors="ignore"), FINAL_PATH)

print("rows:", len(merged), "| columns:", merged.shape[1])
for c in ["in_fixation", "in_saccade", "Pupil_clean"] + aoi_cols:
    if c in merged.columns:
        print(f"  {c:18} coverage {merged[c].notna().mean():.3f}")
print("survey coverage:",
      {c: round(float(merged[c].notna().mean()), 3)
       for c in merged.columns if c.startswith("MDMT")})

  found 55 saccades.csv file(s)
  DROPPED 32,915 saccades from 4 recording(s) not in metadata: ['19d760d0-5bea-4e0a-bc2c-63d6731a29db', '62c6bb29-0cdf-485e-9b8e-defe45592f30', 'a3897410-7936-4edb-9996-10c2089aad22', 'f18ac2ee-e19b-4248-bbf1-08e419d02755']
  176,163 saccades from 25 recording(s), no duration filter applied
 wrote 4,023,825 rows -> /Users/debbiehsu/Documents/PythonProjects/GTRI_MHMUAV/physio_eye_survey.csv
rows: 4023825 | columns: 78
  in_fixation        coverage 0.997
  in_saccade         coverage 0.997
  Pupil_clean        coverage 0.829
  AOI_North          coverage 0.997
  AOI_South          coverage 0.997
  AOI_Face           coverage 0.997
survey coverage: {'MDMT_North_C': 0.997, 'MDMT_South_C': 0.997, 'MDMT_Human_C': 0.997, 'MDMT_North_IS': 0.997, 'MDMT_South_IS': 0.997, 'MDMT_Human_IS': 0.997}
